# CBraMod против DeepEpilepsy в одинаковых условиях + ансамбль двух моделей

| Поле | Значение |
|------|----------|
| **Автор** | Захар Санников (zs) |
| **Дата** | 2026-10-21 |
| **Гипотеза** | Г1 (ЭЭГ) |
| **Цель** | Честно сравнить CBraMod и DeepEpilepsy v2 при одинаковых данных, окнах, аугментациях и без доступа к абсолютной амплитуде; проверить ансамбль двух моделей |
| **Данные** | Обучение: NMT train (Kaggle `mhmdrdwn/nmt-eeg`), до 8 окон по 30 с на запись — **одни и те же окна для обеих моделей**; выбор эпохи — 15 % записей NMT train (одно деление). Проверка один раз в конце: NMT eval и турецкий набор (`buraktaci/turkish-epilepsy`, 50 эпилепсия + 71 здоровый, не виден при обучении) |
| **Амплитуда** | DeepEpilepsy: каждое окно нормируется по каждому каналу (как в v2). **CBraMod: каждое окно приводится к общему СКО 15 мкВ по всем каналам вместе** — соотношение амплитуд между каналами сохраняется, абсолютная амплитуда пропадает. Поэтому общий множитель для NMT (79) больше не нужен |
| **Аугментации (обе модели)** | После нормировки окна: гауссов шум 0,1·СКО (DeepEpilepsy σ = 0,1; CBraMod σ = 1,5 мкВ), зануление 1–2 каналов (p = 0,5), зеркальная замена полушарий (p = 0,5), циклический сдвиг до 2 с |
| **Обучение** | 3 запуска (seed 0, 1, 2), 6 эпох. DeepEpilepsy — как v2 (lr 2·10⁻⁵, пакет 32). CBraMod — как в прошлом блокноте (lr основы 1·10⁻⁴, головы 5·10⁻⁴, косинус, пакет 64). Новых подборов нет |
| **Среда** | Kaggle, GPU T4 |

Прошлые результаты: DeepEpilepsy v2 — ансамбль NMT eval 0,90, Турция 0,73 (0,64–0,81). CBraMod без аугментаций, в абсолютных мкВ — NMT eval 0,91, Турция 0,78 (0,70–0,86), но запуски 0,67 / 0,89 / 0,69, а в турецком наборе у больных выше амплитуда (одна амплитуда даёт 0,79).

## 1. Модели и общие функции

In [ ]:
import os, sys, glob, time, subprocess, importlib.util
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, mne, scipy.io, scipy.signal as ss
from huggingface_hub import hf_hub_download
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata, spearmanr
from concurrent.futures import ThreadPoolExecutor
mne.set_log_level("ERROR")
DEV = "cuda" if torch.cuda.is_available() else "cpu"
T_NB = time.time()

SFREQ, SEG, PATCH = 200, 30, 200
L = SFREQ * SEG

# --- DeepEpilepsy (как в v2) ---
DE_REPO = "/tmp/dl_epilepsy_reeg"
if not os.path.isdir(DE_REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://gitlab.com/chum-epilepsy/dl_epilepsy_reeg.git", DE_REPO], check=True)
spec = importlib.util.spec_from_file_location("de_transformers", f"{DE_REPO}/src/epileptology/ml/models/transformers.py")
de = importlib.util.module_from_spec(spec); spec.loader.exec_module(de)
DE_W = hf_hub_download("emilelemoine/DeepEpilepsy_v1", "DeepEpilepsy_large_50_30s_augment.pt")

def new_de(seed):
    torch.manual_seed(seed)
    m = de.ViT1d(n_class=2, in_channels=19, n_timepoints=L, patch_size=50, tokenizer="conv1d",
                 n_layers=6, n_heads=8, dim_hidden=512, dim_mlp=512, dropout=0.25, attention_dropout=0.25)
    m.load_state_dict(torch.load(DE_W, map_location="cpu", weights_only=True), strict=True)
    return m.to(DEV)

def de_input(W):
    """Нормировка каждого канала окна (как в v2)."""
    x = W.astype(np.float32)
    x = (x - x.mean(-1, keepdims=True)) / (x.std(-1, keepdims=True) + 1e-20)
    return torch.from_numpy(x)

# --- CBraMod ---
CB_REPO = "/tmp/CBraMod"
if not os.path.isdir(CB_REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/wjq-learning/CBraMod.git", CB_REPO], check=True)
sys.path.insert(0, CB_REPO)
from models import cbramod as cbm_mod
from models.cbramod import CBraMod
_pe_forward = cbm_mod.PatchEmbedding.forward
def _pe_forward_fp32(self, x, mask=None):          # БПФ внутри PatchEmbedding не работает в fp16 при длине 200
    with torch.autocast("cuda", enabled=False):
        return _pe_forward(self, x.float(), mask)
cbm_mod.PatchEmbedding.forward = _pe_forward_fp32
CB_PRE = torch.load(hf_hub_download("weighting666/CBraMod", "pretrained_weights.pth"), map_location="cpu", weights_only=True)

class CBNet(nn.Module):
    """CBraMod + голова по всем патчам (all_patch_reps_twolayer). Вход (B, 19, 6000)."""
    def __init__(self, dropout=0.1):
        super().__init__()
        self.backbone = CBraMod(in_dim=200, out_dim=200, d_model=200, dim_feedforward=800, seq_len=30, n_layer=12, nhead=8)
        self.backbone.load_state_dict(CB_PRE, strict=True)
        self.backbone.proj_out = nn.Identity()
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(19 * SEG * 200, 200), nn.ELU(), nn.Dropout(dropout), nn.Linear(200, 1))
    def forward(self, x):
        return self.head(self.backbone(x.reshape(len(x), 19, SEG, PATCH))).squeeze(-1)

def new_cb(seed):
    torch.manual_seed(seed)
    return CBNet().to(DEV)

CB_STD_UV = 15.0
def cb_input(W):
    """Каждое окно → общее СКО 15 мкВ по всем каналам и отсчётам (абсолютная амплитуда пропадает,
    соотношение каналов остаётся), затем мкВ/100 как у авторов CBraMod; обрезка ±5 (±33 СКО) — только от выбросов."""
    x = W.astype(np.float32)
    x = x - x.mean(-1, keepdims=True)
    x = x / (x.std(axis=(1, 2), keepdims=True) + 1e-6) * (CB_STD_UV / 100.0)
    return torch.from_numpy(np.clip(x, -5, 5))

AMP = dict(device_type="cuda", dtype=torch.float16, enabled=DEV == "cuda")

def predict(kind, model, Ws, bs=64):
    """Медиана вероятности по окнам для каждой записи."""
    model.eval(); out = []
    with torch.no_grad(), torch.autocast(**AMP):
        for W in Ws:
            p = []
            for b in range(0, len(W), bs):
                if kind == "de":
                    p.append(torch.softmax(model(de_input(W[b:b + bs]).to(DEV)).float(), 1)[:, 1].cpu().numpy())
                else:
                    p.append(torch.sigmoid(model(cb_input(W[b:b + bs]).to(DEV)).float()).cpu().numpy())
            out.append(np.median(np.concatenate(p)))
    return np.array(out)

def rank(s): return rankdata(s) / len(s)

def boot_ci(yt, s, n=2000, seed=0):
    r = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = r.integers(0, len(yt), len(yt))
        if len(set(yt[i])) == 2: v.append(roc_auc_score(yt[i], s[i]))
    return np.percentile(v, [2.5, 97.5])

def boot_diff(yt, a, b, n=2000, seed=0):
    """Парный бутстреп разности ROC AUC (a − b) на одних и тех же людях."""
    r = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = r.integers(0, len(yt), len(yt))
        if len(set(yt[i])) == 2: v.append(roc_auc_score(yt[i], a[i]) - roc_auc_score(yt[i], b[i]))
    v = np.array(v)
    return np.percentile(v, [2.5, 97.5]), float(np.mean(v <= 0))

BANDS = [(1, 4), (4, 8), (8, 13), (13, 30), (30, 45)]
def rel_powers(W):
    """Относительные мощности ритмов (без амплитуды), среднее по окнам."""
    fr, P = ss.welch(W.astype(np.float32), fs=SFREQ, nperseg=400, axis=-1)
    bp = np.stack([P[..., (fr >= a) & (fr < b)].mean(-1) for a, b in BANDS], -1)
    return (bp / bp.sum(-1, keepdims=True)).mean(0).ravel()

SOS_BP = ss.butter(4, [0.3, 75], btype="bandpass", fs=SFREQ, output="sos")
B_N, A_N = ss.iirnotch(50, 30, fs=SFREQ)
def cb_filter(X):
    """0,3–75 Гц + режекция 50 Гц, как в прошлом блокноте CBraMod."""
    if X.shape[-1] < 60 * SFREQ:
        return X
    return ss.filtfilt(B_N, A_N, ss.sosfiltfilt(SOS_BP, X.astype(np.float64), axis=-1), axis=-1)

def windows(X):
    n = X.shape[1] // L
    return X[:, :n * L].reshape(19, n, L).transpose(1, 0, 2)

# Порядок каналов: читаем в порядке CBraMod, для DeepEpilepsy переставляем в её порядок
EL_CB = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2", "F7", "F8", "T3", "T4", "T5", "T6", "Fz", "Cz", "Pz"]
EL_DE = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "F7", "F8", "T3", "T4", "T5", "T6", "O1", "O2", "Fz", "Cz", "Pz"]
TO_DE = [EL_CB.index(e) for e in EL_DE]
MIRROR_PAIRS = {"Fp1": "Fp2", "F3": "F4", "C3": "C4", "P3": "P4", "O1": "O2", "F7": "F8", "T3": "T4", "T5": "T6"}
MIRROR_PAIRS.update({v: k for k, v in list(MIRROR_PAIRS.items())})
def mirror_idx(el): return [el.index(MIRROR_PAIRS.get(e, e)) for e in el]
MIRROR = {"de": mirror_idx(EL_DE), "cb": mirror_idx(EL_CB)}
print("устройство:", torch.cuda.get_device_name(0) if DEV == "cuda" else "процессор")

## 2. NMT: подкачка файлов, чтение (одни и те же окна для обеих моделей)

In [ ]:
ROOT_NMT = glob.glob("/kaggle/input/**/nmt_scalp_eeg_dataset", recursive=True)[0]
lab = pd.read_csv(f"{ROOT_NMT}/Labels.csv")
paths = {os.path.basename(p): p for p in glob.glob(f"{ROOT_NMT}/**/*.edf", recursive=True)}
lab["path"] = lab["recordname"].map(paths)
lab = lab.dropna(subset=["path"]).sort_values("recordname").reset_index(drop=True)
lab["y"] = (lab["label"].str.lower() == "abnormal").astype(int)
print(pd.crosstab(lab["loc"], lab["label"]))

def warm(p):
    with open(p, "rb") as f:
        while f.read(1 << 24):
            pass
t0 = time.time()
with ThreadPoolExecutor(32) as ex:
    for i, _ in enumerate(ex.map(warm, lab.path)):
        if (i + 1) % 400 == 0 or i + 1 == len(lab):
            print(f"NMT подкачано {i+1}/{len(lab)} {time.time()-t0:.0f} с", flush=True)

ALIAS = {"T7": "T3", "T8": "T4", "P7": "T5", "P8": "T6"}
TARGET = [e.upper() for e in EL_CB]
def norm(ch):
    c = ch.upper().replace("EEG", "").strip()
    for s in ["-REF", "-AVG", "-LE", "-AR"]:
        c = c.replace(s, "")
    return ALIAS.get(c.strip(" -_"), c.strip(" -_"))

# Масштаб NMT в файлах неверный (СКО ≈0,2 мкВ). Обе модели нормируют каждое окно, поэтому абсолютный масштаб
# на них не влияет; множитель 100 — только чтобы значения хорошо хранились в float16.
NMT_STORE_K = 100.0
def nmt_job(i):
    r = lab.iloc[i]
    try:
        raw = mne.io.read_raw_edf(r.path, preload=True)
        if raw.info["sfreq"] != SFREQ:
            raw.resample(SFREQ)
        names = {norm(c): c for c in raw.ch_names}
        X = raw.get_data(picks=[names[t] for t in TARGET]) * 1e6 * NMT_STORE_K
    except Exception:
        return r.recordname, None
    W_de, W_cb = windows(X)[:, TO_DE], windows(cb_filter(X))
    if len(W_de) == 0:
        return r.recordname, None
    if r["loc"] == "train" and len(W_de) > 8:
        k = np.sort(np.random.default_rng(i).choice(len(W_de), 8, replace=False))
        W_de, W_cb = W_de[k], W_cb[k]
    return r.recordname, (W_de.astype(np.float16), W_cb.astype(np.float16), rel_powers(W_de),
                          float(np.std(W_de)) / NMT_STORE_K)

NMT_DE, NMT_CB, NMT_BP, NMT_AMP = {}, {}, {}, {}
t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    for i, (name, res) in enumerate(ex.map(nmt_job, range(len(lab)))):
        if res is not None:
            NMT_DE[name], NMT_CB[name], NMT_BP[name], NMT_AMP[name] = res
        if (i + 1) % 400 == 0 or i + 1 == len(lab):
            print(f"NMT прочитано {i+1}/{len(lab)} {time.time()-t0:.0f} с", flush=True)
lab = lab[lab.recordname.isin(NMT_DE)].reset_index(drop=True)
tr, ev = lab[lab["loc"] == "train"].reset_index(drop=True), lab[lab["loc"] == "eval"].reset_index(drop=True)
print(pd.crosstab(lab["loc"], lab["label"]))
print("окон: train", sum(len(NMT_DE[n]) for n in tr.recordname), "| eval", sum(len(NMT_DE[n]) for n in ev.recordname))

## 3. Турецкий набор: 19 каналов, средний референт, 200 Гц, окна 30 с (DeepEpilepsy — без фильтра, как в v2; CBraMod — с фильтром, как раньше)

In [ ]:
ROOT_TR = glob.glob("/kaggle/input/**/Turkish Epilepsy Dataset", recursive=True)[0]
tfiles = [(f, 1) for f in sorted(glob.glob(f"{ROOT_TR}/epilepsy/*.mat"))] + \
         [(f, 0) for f in sorted(glob.glob(f"{ROOT_TR}/normal/*.mat"))]
t0 = time.time()
with ThreadPoolExecutor(16) as ex:
    for i, _ in enumerate(ex.map(warm, [f for f, _ in tfiles])):
        if (i + 1) % 40 == 0 or i + 1 == len(tfiles):
            print(f"Турция подкачано {i+1}/{len(tfiles)} {time.time()-t0:.0f} с", flush=True)
# индексы турецких каналов в порядке CBraMod: Fp1 Fp2 F3 F4 C3 C4 P3 P4 O1  O2  F7  F8  T3  T4  T5  T6  Fz Cz Pz
IDX = [0, 1, 2, 3, 5, 6, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 4, 7, 10]

def tr_job(f):
    d = scipy.io.loadmat(f)["data"]
    X = np.stack([np.stack([d[s, c].ravel() for c in IDX]) for s in range(d.shape[0])])   # (куски, 19, 7500)
    X = X - X.mean(1, keepdims=True)                                                       # средний референт
    X = ss.resample_poly(X, 2, 5, axis=-1)                                                  # 200 Гц
    n = len(X) // 2
    W_raw = np.concatenate([X[0:2*n:2], X[1:2*n:2]], axis=-1)                              # (окна, 19, 6000), как в v2
    W_cb = windows(cb_filter(np.concatenate(list(X), axis=-1)))
    return W_raw[:, TO_DE].astype(np.float16), W_cb.astype(np.float16), rel_powers(W_raw), float(np.std(W_raw))

TR_DE, TR_CB, TR_BP, TR_AMP = [], [], [], []
t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    for i, (a, b, c, d) in enumerate(ex.map(tr_job, [f for f, _ in tfiles])):
        TR_DE.append(a); TR_CB.append(b); TR_BP.append(c); TR_AMP.append(d)
y_tr = np.array([yy for _, yy in tfiles]); tr_ids = [os.path.basename(f) for f, _ in tfiles]
TR_AMP = np.array(TR_AMP)
print(f"Турция прочитано за {time.time()-t0:.0f} с | эпилепсия {int(y_tr.sum())}, здоровые {int((1-y_tr).sum())} | окон на запись: медиана {int(np.median([len(w) for w in TR_DE]))}")
print("эталон перекоса: СКО сигнала записи как оценка эпилепсии на Турции, ROC AUC =", round(roc_auc_score(y_tr, TR_AMP), 3))

## 4. Дообучение: CBraMod и DeepEpilepsy, по 3 запуска

Деление NMT train на обучение / отложенные 15 % одно для обеих моделей и всех запусков. Эпоха выбирается по ROC AUC на отложенных записях. NMT eval и Турция считаются только после обучения, по ним ничего не выбирается.

In [ ]:
from sklearn.model_selection import train_test_split

fit_names, val_names = train_test_split(tr.recordname.to_numpy(), test_size=0.15, stratify=tr.y, random_state=0)
ymap = dict(zip(lab.recordname, lab.y))
yw = np.concatenate([np.full(len(NMT_DE[n]), ymap[n]) for n in fit_names])
y_val = np.array([ymap[n] for n in val_names])
w_cls = np.where(yw == 1, 0.5 / (yw == 1).sum(), 0.5 / (yw == 0).sum()); w_cls /= w_cls.sum()
print(f"окон для обучения: {len(yw)} (патология {yw.mean():.2f}) | записей на выбор эпохи: {len(val_names)}")

NOISE = {"de": 0.1, "cb": 0.1 * CB_STD_UV / 100.0}   # 0,1 СКО окна в масштабе входа каждой модели
def augment(x, g, kind):
    """x: (B, 19, L) после нормировки, на GPU. Одинаковые аугментации для обеих моделей."""
    B = x.shape[0]
    x = x + NOISE[kind] * torch.randn_like(x)
    m = torch.rand(B, device=x.device) < 0.5
    x[m] = x[m][:, MIRROR[kind]]
    for b in range(B):
        if g.random() < 0.5:
            x[b, g.choice(19, g.integers(1, 3), replace=False)] = 0
        x[b] = torch.roll(x[b], int(g.integers(-2 * SFREQ, 2 * SFREQ + 1)), dims=-1)
    return x

def train_run(kind, seed, Xw, epochs=6):
    g = np.random.default_rng(seed)
    if kind == "de":
        model, bs = new_de(seed), 32
        opt = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.05); sched = None
        to_in = de_input
        lossf = lambda out, t: F.cross_entropy(out.float(), t.long())
    else:
        model, bs = new_cb(seed), 64
        back = [p for n, p in model.named_parameters() if n.startswith("backbone")]
        other = [p for n, p in model.named_parameters() if not n.startswith("backbone")]
        opt = torch.optim.AdamW([{"params": back, "lr": 1e-4}, {"params": other, "lr": 1e-3 * (bs / 256) ** 0.5}], weight_decay=0.05)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * int(np.ceil(len(yw) / bs)), eta_min=1e-6)
        to_in = cb_input
        lossf = lambda out, t: F.binary_cross_entropy_with_logits(out.float(), t.float())
    scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
    val_W = [(NMT_DE if kind == "de" else NMT_CB)[n] for n in val_names]
    best_auc, best_state, best_ep, hist, n_bad = -1, None, 0, [], 0
    t0 = time.time()
    for ep in range(epochs):
        model.train(); losses = []
        idx = g.choice(len(yw), size=len(yw), replace=True, p=w_cls)
        for b in range(0, len(idx), bs):
            j = idx[b:b + bs]
            x = augment(to_in(Xw[j]).to(DEV), g, kind)
            t = torch.tensor(yw[j], device=DEV)
            with torch.autocast(**AMP):
                out = model(x)
            loss = lossf(out, t)
            if not torch.isfinite(loss):
                n_bad += 1
            opt.zero_grad(); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
            if sched: sched.step()
            losses.append(loss.item())
        auc = roc_auc_score(y_val, predict(kind, model, val_W))
        hist.append((round(float(np.nanmean(losses)), 3), round(auc, 3)))
        print(f"  {kind} запуск {seed} | эпоха {ep+1}: потери {np.nanmean(losses):.3f} | отложенные NMT {auc:.3f} | шагов с NaN/inf {n_bad} | {time.time()-t0:.0f} с", flush=True)
        if auc > best_auc:
            best_auc, best_ep, best_state = auc, ep + 1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, best_ep, best_auc, hist, n_bad

TIME_LIMIT_MIN = 85    # всё обучение; запуск, который не уложится, пропускается (с сообщением)
RUNS, T_TRAIN = [], time.time()
for kind, store in [("cb", NMT_CB), ("de", NMT_DE)]:
    Xw = np.concatenate([store[n] for n in fit_names])
    dur = []
    for seed in [0, 1, 2]:
        est = np.mean(dur) if dur else (15 if kind == "cb" else 8) * 60
        if (time.time() - T_TRAIN + est) / 60 > TIME_LIMIT_MIN:
            print(f"ПРОПУСК {kind} запуск {seed}: не укладывается в {TIME_LIMIT_MIN} мин"); continue
        t0 = time.time()
        model, best_ep, best_auc, hist, n_bad = train_run(kind, seed, Xw)
        s_ev = predict(kind, model, [store[n] for n in ev.recordname])
        s_tr = predict(kind, model, TR_DE if kind == "de" else TR_CB)
        dur.append(time.time() - t0)
        RUNS.append(dict(model=kind, seed=seed, best_epoch=best_ep, val=round(best_auc, 3), hist=hist, nan_steps=n_bad,
                         nmt_eval=roc_auc_score(ev.y, s_ev), turkey=roc_auc_score(y_tr, s_tr), s_ev=s_ev, s_tr=s_tr,
                         turkey_med_p0=float(np.median(s_tr[y_tr == 0])), turkey_med_p1=float(np.median(s_tr[y_tr == 1])),
                         n_unique_tr=len(np.unique(s_tr)), minutes=round(dur[-1] / 60, 1)))
        print(f"{kind} запуск {seed}: эпоха {best_ep} | NMT eval {RUNS[-1]['nmt_eval']:.3f} | Турция {RUNS[-1]['turkey']:.3f} | "
              f"медиана p здоровые/эпилепсия {RUNS[-1]['turkey_med_p0']:.3f}/{RUNS[-1]['turkey_med_p1']:.3f} | "
              f"разных значений {RUNS[-1]['n_unique_tr']}/{len(s_tr)} | {dur[-1]/60:.1f} мин", flush=True)
        del model; torch.cuda.empty_cache()
    del Xw
print(f"всё обучение: {(time.time()-T_TRAIN)/60:.1f} мин")
runs = pd.DataFrame([{k: v for k, v in r.items() if k not in ("s_ev", "s_tr")} for r in RUNS])
runs.to_csv("/kaggle/working/cbramod_vs_deepepilepsy_runs.csv", index=False)
print(runs.drop(columns="hist").round(3).to_string(index=False))

## 5. Итоги: запуски, ансамбли каждой модели, ансамбль двух моделей, эталон

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

y_ev = ev.y.to_numpy()
# эталон: относительные мощности ритмов, обучение на NMT train
clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.01, max_iter=5000, class_weight="balanced"))
clf.fit(np.stack([NMT_BP[n] for n in tr.recordname]), tr.y)
bp_ev = clf.predict_proba(np.stack([NMT_BP[n] for n in ev.recordname]))[:, 1]
bp_tr = clf.predict_proba(np.stack(TR_BP))[:, 1]

ENS = {}
for kind in ["cb", "de"]:
    rr = [r for r in RUNS if r["model"] == kind]
    ENS[kind] = (np.mean([rank(r["s_ev"]) for r in rr], 0), np.mean([rank(r["s_tr"]) for r in rr], 0))
ENS["both"] = ((ENS["cb"][0] + ENS["de"][0]) / 2, (ENS["cb"][1] + ENS["de"][1]) / 2)
ENS["bp"] = (bp_ev, bp_tr)

NAMES = {"cb": "CBraMod (нормировка окна + аугм.)", "de": "DeepEpilepsy v2", "both": "Ансамбль CBraMod + DeepEpilepsy",
         "bp": "Относительные мощности ритмов"}
def fmt(y, s):
    lo, hi = boot_ci(y, s); return f"{roc_auc_score(y, s):.3f} ({lo:.2f}–{hi:.2f})"
rows = []
for kind in ["cb", "de"]:
    rr = runs[runs.model == kind]
    rows.append({"модель": NAMES[kind] + ", запуски", "n": len(rr),
                 "NMT eval": f"{rr.nmt_eval.mean():.3f} ± {rr.nmt_eval.std(ddof=1):.3f}",
                 "Турция": f"{rr.turkey.mean():.3f} ± {rr.turkey.std(ddof=1):.3f} ({' / '.join(f'{v:.2f}' for v in rr.turkey)})"})
for kind in ["cb", "de", "both", "bp"]:
    rows.append({"модель": NAMES[kind] + (", ансамбль запусков" if kind in ("cb", "de") else ""), "n": "",
                 "NMT eval": fmt(y_ev, ENS[kind][0]), "Турция": fmt(y_tr, ENS[kind][1])})
summary = pd.DataFrame(rows)
summary.to_csv("/kaggle/working/cbramod_vs_deepepilepsy_summary.csv", index=False)
print(summary.to_string(index=False))

print("\nПарный бутстреп разности ROC AUC (95% ДИ, доля повторов с разностью ≤ 0):")
for a, b in [("cb", "de"), ("both", "de"), ("both", "cb")]:
    for nm, y, i in [("NMT eval", y_ev, 0), ("Турция", y_tr, 1)]:
        (lo, hi), p = boot_diff(y, ENS[a][i], ENS[b][i])
        print(f"  {NAMES[a]} − {NAMES[b]}, {nm}: {roc_auc_score(y, ENS[a][i]) - roc_auc_score(y, ENS[b][i]):+.3f} ({lo:+.3f}…{hi:+.3f}), p≤0: {p:.2f}")

print("\nСвязь с амплитудой записи на Турции (Спирмен; у больных амплитуда выше):")
for kind in ["cb", "de", "both", "bp"]:
    print(f"  {NAMES[kind]}: {spearmanr(ENS[kind][1], TR_AMP).correlation:+.2f}")
print("Согласие CBraMod и DeepEpilepsy (Спирмен): NMT eval", round(spearmanr(ENS['cb'][0], ENS['de'][0]).correlation, 2),
      "| Турция", round(spearmanr(ENS['cb'][1], ENS['de'][1]).correlation, 2))

# предсказания по людям — для будущих ансамблей
pp = []
for ds, ids, y, i in [("nmt_eval", ev.recordname.tolist(), y_ev, 0), ("turkey", tr_ids, y_tr, 1)]:
    d = pd.DataFrame({"dataset": ds, "id": ids, "y": y})
    for r in RUNS:
        d[f"{r['model']}_seed{r['seed']}"] = r["s_ev" if i == 0 else "s_tr"]
    for k in ["cb", "de", "both"]:
        d[f"{k}_ens_rank"] = ENS[k][i]
    d["relpow"] = ENS["bp"][i]
    pp.append(d)
pd.concat(pp).to_csv("/kaggle/working/cbramod_vs_deepepilepsy_per_person.csv", index=False)
print(f"\nвсё время блокнота: {(time.time()-T_NB)/60:.1f} мин")

## 6. Вывод Kaggle (без строк о подкачке файлов)

Тестовые наборы считались один раз, по ним ничего не подбиралось. NaN/inf нет, потери падают плавно, у групп вероятности различны (117–119 разных значений из 121).

```
устройство: Tesla T4
label  abnormal  normal
loc                    
eval         90      95
train       325    1907
label  abnormal  normal
loc                    
eval         90      94
train       323    1902
окон: train 17643 | eval 4331
Турция прочитано за 129 с | эпилепсия 50, здоровые 71 | окон на запись: медиана 37
эталон перекоса: СКО сигнала записи как оценка эпилепсии на Турции, ROC AUC = 0.763
окон для обучения: 15007 (патология 0.14) | записей на выбор эпохи: 334
  cb запуск 0 | эпоха 1: потери 0.686 | отложенные NMT 0.792 | шагов с NaN/inf 0 | 143 с
  cb запуск 0 | эпоха 2: потери 0.583 | отложенные NMT 0.844 | шагов с NaN/inf 0 | 288 с
  cb запуск 0 | эпоха 3: потери 0.510 | отложенные NMT 0.905 | шагов с NaN/inf 0 | 436 с
  cb запуск 0 | эпоха 4: потери 0.464 | отложенные NMT 0.898 | шагов с NaN/inf 0 | 585 с
  cb запуск 0 | эпоха 5: потери 0.436 | отложенные NMT 0.899 | шагов с NaN/inf 0 | 733 с
  cb запуск 0 | эпоха 6: потери 0.419 | отложенные NMT 0.901 | шагов с NaN/inf 0 | 881 с
cb запуск 0: эпоха 3 | NMT eval 0.827 | Турция 0.782 | медиана p здоровые/эпилепсия 0.003/0.018 | разных значений 118/121 | 15.3 мин
  cb запуск 1 | эпоха 1: потери 0.669 | отложенные NMT 0.812 | шагов с NaN/inf 0 | 148 с
  cb запуск 1 | эпоха 2: потери 0.570 | отложенные NMT 0.858 | шагов с NaN/inf 0 | 297 с
  cb запуск 1 | эпоха 3: потери 0.518 | отложенные NMT 0.880 | шагов с NaN/inf 0 | 446 с
  cb запуск 1 | эпоха 4: потери 0.474 | отложенные NMT 0.894 | шагов с NaN/inf 0 | 594 с
  cb запуск 1 | эпоха 5: потери 0.432 | отложенные NMT 0.895 | шагов с NaN/inf 0 | 743 с
  cb запуск 1 | эпоха 6: потери 0.411 | отложенные NMT 0.896 | шагов с NaN/inf 0 | 891 с
cb запуск 1: эпоха 6 | NMT eval 0.830 | Турция 0.810 | медиана p здоровые/эпилепсия 0.000/0.022 | разных значений 119/121 | 15.4 мин
  cb запуск 2 | эпоха 1: потери 0.754 | отложенные NMT 0.784 | шагов с NaN/inf 0 | 148 с
  cb запуск 2 | эпоха 2: потери 0.622 | отложенные NMT 0.832 | шагов с NaN/inf 0 | 296 с
  cb запуск 2 | эпоха 3: потери 0.572 | отложенные NMT 0.812 | шагов с NaN/inf 0 | 445 с
  cb запуск 2 | эпоха 4: потери 0.533 | отложенные NMT 0.860 | шагов с NaN/inf 0 | 593 с
  cb запуск 2 | эпоха 5: потери 0.484 | отложенные NMT 0.877 | шагов с NaN/inf 0 | 741 с
  cb запуск 2 | эпоха 6: потери 0.454 | отложенные NMT 0.886 | шагов с NaN/inf 0 | 890 с
cb запуск 2: эпоха 6 | NMT eval 0.851 | Турция 0.833 | медиана p здоровые/эпилепсия 0.003/0.022 | разных значений 117/121 | 15.4 мин
  de запуск 0 | эпоха 1: потери 0.622 | отложенные NMT 0.829 | шагов с NaN/inf 0 | 39 с
  de запуск 0 | эпоха 2: потери 0.556 | отложенные NMT 0.842 | шагов с NaN/inf 0 | 78 с
  de запуск 0 | эпоха 3: потери 0.524 | отложенные NMT 0.859 | шагов с NaN/inf 0 | 117 с
  de запуск 0 | эпоха 4: потери 0.514 | отложенные NMT 0.858 | шагов с NaN/inf 0 | 155 с
  de запуск 0 | эпоха 5: потери 0.498 | отложенные NMT 0.859 | шагов с NaN/inf 0 | 194 с
  de запуск 0 | эпоха 6: потери 0.483 | отложенные NMT 0.858 | шагов с NaN/inf 0 | 233 с
de запуск 0: эпоха 5 | NMT eval 0.888 | Турция 0.686 | медиана p здоровые/эпилепсия 0.748/0.844 | разных значений 118/121 | 4.1 мин
  de запуск 1 | эпоха 1: потери 0.618 | отложенные NMT 0.826 | шагов с NaN/inf 0 | 39 с
  de запуск 1 | эпоха 2: потери 0.560 | отложенные NMT 0.837 | шагов с NaN/inf 0 | 77 с
  de запуск 1 | эпоха 3: потери 0.526 | отложенные NMT 0.839 | шагов с NaN/inf 0 | 116 с
  de запуск 1 | эпоха 4: потери 0.515 | отложенные NMT 0.835 | шагов с NaN/inf 0 | 154 с
  de запуск 1 | эпоха 5: потери 0.499 | отложенные NMT 0.851 | шагов с NaN/inf 0 | 193 с
  de запуск 1 | эпоха 6: потери 0.479 | отложенные NMT 0.865 | шагов с NaN/inf 0 | 231 с
de запуск 1: эпоха 6 | NMT eval 0.900 | Турция 0.674 | медиана p здоровые/эпилепсия 0.189/0.307 | разных значений 119/121 | 4.0 мин
  de запуск 2 | эпоха 1: потери 0.622 | отложенные NMT 0.815 | шагов с NaN/inf 0 | 38 с
  de запуск 2 | эпоха 2: потери 0.561 | отложенные NMT 0.853 | шагов с NaN/inf 0 | 77 с
  de запуск 2 | эпоха 3: потери 0.529 | отложенные NMT 0.842 | шагов с NaN/inf 0 | 115 с
  de запуск 2 | эпоха 4: потери 0.509 | отложенные NMT 0.851 | шагов с NaN/inf 0 | 154 с
  de запуск 2 | эпоха 5: потери 0.498 | отложенные NMT 0.857 | шагов с NaN/inf 0 | 193 с
  de запуск 2 | эпоха 6: потери 0.482 | отложенные NMT 0.859 | шагов с NaN/inf 0 | 231 с
de запуск 2: эпоха 6 | NMT eval 0.895 | Турция 0.682 | медиана p здоровые/эпилепсия 0.347/0.479 | разных значений 119/121 | 4.0 мин
всё обучение: 58.3 мин
model  seed  best_epoch   val  nan_steps  nmt_eval  turkey  turkey_med_p0  turkey_med_p1  n_unique_tr  minutes
   cb     0           3 0.905          0     0.827   0.782          0.003          0.018          118     15.3
   cb     1           6 0.896          0     0.830   0.810          0.000          0.022          119     15.4
   cb     2           6 0.886          0     0.851   0.833          0.003          0.022          117     15.4
   de     0           5 0.859          0     0.888   0.686          0.748          0.844          118      4.1
   de     1           6 0.865          0     0.900   0.674          0.189          0.307          119      4.0
   de     2           6 0.859          0     0.895   0.682          0.347          0.479          119      4.0
                                              модель n          NMT eval                             Турция
          CBraMod (нормировка окна + аугм.), запуски 3     0.836 ± 0.013 0.808 ± 0.025 (0.78 / 0.81 / 0.83)
                            DeepEpilepsy v2, запуски 3     0.894 ± 0.006 0.680 ± 0.006 (0.69 / 0.67 / 0.68)
CBraMod (нормировка окна + аугм.), ансамбль запусков   0.845 (0.79–0.90)                  0.820 (0.74–0.88)
                  DeepEpilepsy v2, ансамбль запусков   0.900 (0.85–0.94)                  0.682 (0.59–0.77)
                     Ансамбль CBraMod + DeepEpilepsy   0.893 (0.85–0.93)                  0.790 (0.71–0.86)
                       Относительные мощности ритмов   0.822 (0.76–0.88)                  0.576 (0.46–0.69)

Парный бутстреп разности ROC AUC (95% ДИ, доля повторов с разностью ≤ 0):
  CBraMod (нормировка окна + аугм.) − DeepEpilepsy v2, NMT eval: -0.054 (-0.104…-0.009), p≤0: 0.99
  CBraMod (нормировка окна + аугм.) − DeepEpilepsy v2, Турция: +0.138 (+0.034…+0.238), p≤0: 0.01
  Ансамбль CBraMod + DeepEpilepsy − DeepEpilepsy v2, NMT eval: -0.007 (-0.030…+0.016), p≤0: 0.73
  Ансамбль CBraMod + DeepEpilepsy − DeepEpilepsy v2, Турция: +0.108 (+0.048…+0.171), p≤0: 0.00
  Ансамбль CBraMod + DeepEpilepsy − CBraMod (нормировка окна + аугм.), NMT eval: +0.047 (+0.020…+0.077), p≤0: 0.00
  Ансамбль CBraMod + DeepEpilepsy − CBraMod (нормировка окна + аугм.), Турция: -0.030 (-0.084…+0.023), p≤0: 0.88

Связь с амплитудой записи на Турции (Спирмен; у больных амплитуда выше):
  CBraMod (нормировка окна + аугм.): +0.12
  DeepEpilepsy v2: +0.12
  Ансамбль CBraMod + DeepEpilepsy: +0.14
  Относительные мощности ритмов: +0.17
Согласие CBraMod и DeepEpilepsy (Спирмен): NMT eval 0.82 | Турция 0.4

всё время блокнота: 67.1 мин
```

## 7. Выводы

| Модель | NMT eval (95% ДИ) | Турция (95% ДИ) | Запуски на Турции |
|---|---|---|---|
| CBraMod, нормировка окна + аугм., ансамбль 3 | 0,845 (0,79–0,90) | **0,820 (0,74–0,88)** | 0,78 / 0,81 / 0,83 (0,81 ± 0,03) |
| DeepEpilepsy v2, ансамбль 3 | **0,900 (0,85–0,94)** | 0,682 (0,59–0,77) | 0,69 / 0,67 / 0,68 (0,68 ± 0,01) |
| Ансамбль двух моделей | 0,893 (0,85–0,93) | 0,790 (0,71–0,86) | — |
| Относительные мощности ритмов | 0,822 (0,76–0,88) | 0,576 (0,46–0,69) | — |

1. **Нормировка амплитуды исправила разброс CBraMod**: на Турции запуски 0,78–0,83 (раньше 0,67–0,89), корреляция оценок с амплитудой записи +0,12 (одна амплитуда даёт 0,76). Значит, 0,82 — не от амплитуды. Потери на обучении теперь 0,41–0,45, а не 0,02: аугментации убрали запоминание.
2. **Модели меняются местами**: на NMT eval лучше DeepEpilepsy (0,90 против 0,85; разность по парному бутстрепу 0,05, ДИ 0,01–0,10), на Турции лучше CBraMod (0,82 против 0,68; разность 0,14, ДИ 0,03–0,24). Задача «эпилепсия против здоровых» ближе к цели Г1, чем «патология», но Турция — 121 человек из одного источника.
3. **Ансамбль**: к DeepEpilepsy добавляет много (Турция +0,11), но CBraMod в одиночку на Турции не хуже (ансамбль 0,79 против 0,82, разность −0,03, ДИ −0,08…+0,02 — не отличается), зато на NMT eval ансамбль лучше CBraMod (+0,05). Оценки моделей согласуются слабо (Спирмен 0,40 на Турции), то есть они видят разное.
4. DeepEpilepsy на Турции здесь 0,68, а в v2 было 0,73 (0,72 ± 0,01 по запускам). Настройки те же, но данные чуть другие (случайный выбор окон NMT, перестановка каналов, пересчёт признаков); сама разница лежит в пределах ДИ. Эталон по мощностям ритмов тоже ниже прежнего (0,58 против 0,64–0,66) — оценка на Турции шумная.

**Что брать основной:** CBraMod с нормировкой окна и аугментациями (лучше переносится на «эпилепсия/здоровые» и стабильна по запускам). Ансамбль с DeepEpilepsy — запасной вариант: на Турции не лучше, на NMT выигрывает у CBraMod, и он устойчивее к тому, какая из двух моделей хуже на новом наборе.

**Оговорки:** 3 запуска; Турция — 121 человек, ДИ широкие (0,74–0,88 и 0,59–0,77 пересекаются по краям; разность значима по парному бутстрепу); NMT («патология») и Турция («эпилепсия») — разные задачи; CBraMod предобучена на TUEG, для проверки на TUEP нужно исключить пересечение пациентов; неотфильтрованные для DeepEpilepsy и отфильтрованные для CBraMod входы — как в прошлых блокнотах.

**Затраты:** 67 мин GPU T4 (обучение 58 мин).

Файлы: `results/cbramod_vs_deepepilepsy_{runs,summary,per_person}.csv`.